In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('06_hechos')

# 06 - Tabla de hechos de ventas

Construye y carga `dw.fact_ventas` con las columnas del modelo Kimball existente en `WideWorldImportersDW2`.

**Grano:** una fila por detalle de factura. `numero_factura = InvoiceID` y `numero_linea = InvoiceLineID`, sin renumerar al excluir rechazos.

**Dimensiones:** `dw.dim_tiempo`, `dw.dim_cliente`, `dw.dim_producto`, `dw.dim_vendedor` y `dw.dim_ciudad`.

**Antes de ejecutar:** cargar las dimensiones con `../01_kimball_create_DIM.ipynb` y ejecutar extraccion, profiling, cuarentena y staging de ambas tablas. Se necesitan `staging.invoice_lines` y `staging.invoices`.

Este notebook si conecta a SQL Server e inserta hechos reales. No borra ni recrea una tabla existente, ni modifica los notebooks de referencia.

In [ ]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_DESTINO
from sqlalchemy import text

engine_origen, engine_destino = obtener_engines()
print(f'Destino de la tabla de hechos: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas correctamente.
Servidor: localhost\SQLEXPRESS02
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


## 1. Preparar medidas y resolver claves dimensionales

Las filas deben existir en staging y no tener cuarentena ni errores criticos. RAW solo completa `InvoiceLineID`, `UnitPrice` y `LineProfit` desde el payload del mismo detalle aceptado; no se extraen ventas directamente del OLTP.

Cliente y producto se resuelven por fecha de factura y vigencia SCD2, incluyendo el fin abierto `9999-12-31`. Como en el notebook de referencia, `PERMITIR_VERSION_ACTUAL = True` permite usar la version actual si no hay historia para esa fecha; se informa cuantas filas usan ese fallback. Cambiarlo a `False` exige coincidencia historica estricta.

Vendedor se resuelve por NK. Ciudad se obtiene de la geografia de la version de cliente resuelta (`ciudad`, `estado_provincia`, `pais`). Una geografia sin coincidencia o ambigua no recibe una SK inventada. Ninguna fila sin las cinco claves se publica.

In [ ]:
import json
from datetime import date
from decimal import Decimal
import pandas as pd


def preparar_ventas(fuente):
    ventas = fuente.copy()
    payloads = ventas['payload_linea'].map(json.loads)
    ventas['numero_linea'] = payloads.map(lambda payload: payload.get('InvoiceLineID'))
    ventas['precio_unitario'] = payloads.map(lambda payload: payload.get('UnitPrice'))
    ventas['margen_bruto'] = payloads.map(lambda payload: payload.get('LineProfit'))
    for campo in ['numero_linea', 'numero_factura', 'id_producto_nk']:
        ventas[campo] = pd.to_numeric(ventas[campo], errors='raise').astype('Int64')
        if ventas[campo].isna().any() or (ventas[campo] <= 0).any():
            raise ValueError(f'Clave de negocio invalida: {campo}')
    ventas = ventas.sort_values('id_raw_linea').drop_duplicates('numero_linea', keep='last')
    sin_cabecera = ventas.loc[ventas['fecha_factura'].isna()].copy()
    ventas = ventas.loc[ventas['fecha_factura'].notna()].copy()
    ventas['fecha_factura'] = pd.to_datetime(ventas['fecha_factura'], errors='raise')
    ventas['id_tiempo'] = ventas['fecha_factura'].dt.strftime('%Y%m%d').astype(int)
    for campo in ['id_cliente_nk', 'id_vendedor_nk']:
        ventas[campo] = pd.to_numeric(ventas[campo], errors='raise').astype('Int64')
    for campo in ['cantidad', 'precio_unitario', 'monto_neto', 'monto_impuesto', 'monto_total', 'margen_bruto']:
        valores = []
        for valor in ventas[campo]:
            if pd.isna(valor):
                numero = None
            else:
                numero = Decimal(str(valor))
                if not numero.is_finite():
                    raise ValueError(f'Medida no finita en {campo}')
            if numero is None and campo != 'margen_bruto':
                raise ValueError(f'Medida requerida sin valor: {campo}')
            valores.append(numero)
        ventas[campo] = pd.Series(valores, index=ventas.index, dtype=object)
    ventas['costo_estimado'] = pd.Series([
        neto - margen if margen is not None else None
        for neto, margen in zip(ventas['monto_neto'], ventas['margen_bruto'])
    ], index=ventas.index, dtype=object)
    if any(neto + impuesto != total for neto, impuesto, total in zip(
        ventas['monto_neto'], ventas['monto_impuesto'], ventas['monto_total']
    )):
        raise ValueError('No se cumple neto + impuesto = total')
    ventas['_row_id'] = range(len(ventas))
    return ventas, sin_cabecera


def resolver_scd2(ventas, dimension, clave_natural, clave_sustituta, permitir_actual=True):
    candidatos = ventas[['_row_id', clave_natural, 'fecha_factura']].merge(
        dimension, on=clave_natural, how='left'
    )
    for campo in ['fecha_inicio_vigencia', 'fecha_fin_vigencia']:
        candidatos[campo] = candidatos[campo].map(
            lambda valor: date.fromisoformat(str(valor)[:10]) if pd.notna(valor) else date.min
        )
    fechas = candidatos['fecha_factura'].dt.date
    vigente = candidatos.loc[
        candidatos[clave_sustituta].notna()
        & fechas.ge(candidatos['fecha_inicio_vigencia'])
        & fechas.le(candidatos['fecha_fin_vigencia'])
    ]
    if vigente['_row_id'].duplicated().any():
        raise ValueError(f'Vigencias SCD2 superpuestas en {clave_natural}')
    actual = candidatos.iloc[0:0]
    if permitir_actual:
        actual = candidatos.loc[
            ~candidatos['_row_id'].isin(vigente['_row_id'])
            & candidatos[clave_sustituta].notna()
            & candidatos['es_actual'].eq(1)
        ]
        if actual['_row_id'].duplicated().any():
            raise ValueError(f'Multiples versiones actuales en {clave_natural}')
    resueltas = pd.concat([vigente, actual], ignore_index=True)
    return resueltas[['_row_id', clave_sustituta]], len(actual)


def construir_hechos(ventas, tiempo, clientes, productos, vendedores, ciudades, permitir_actual=True):
    claves_cliente, fallback_cliente = resolver_scd2(
        ventas, clientes, 'id_cliente_nk', 'id_cliente_sk', permitir_actual
    )
    claves_producto, fallback_producto = resolver_scd2(
        ventas, productos, 'id_producto_nk', 'id_producto_sk', permitir_actual
    )
    hechos = ventas.merge(tiempo[['id_tiempo']], on='id_tiempo', how='left',
                          validate='many_to_one', indicator='_tiempo')
    hechos.loc[hechos['_tiempo'] != 'both', 'id_tiempo'] = pd.NA
    hechos = hechos.merge(claves_cliente, on='_row_id', how='left', validate='one_to_one')
    hechos = hechos.merge(claves_producto, on='_row_id', how='left', validate='one_to_one')
    hechos = hechos.merge(vendedores, on='id_vendedor_nk', how='left', validate='many_to_one')
    geografia = ['ciudad', 'estado_provincia', 'pais']
    hechos = hechos.merge(clientes[['id_cliente_sk'] + geografia],
                          on='id_cliente_sk', how='left', validate='many_to_one')
    ciudades_validas = ciudades.dropna(subset=geografia)
    hechos = hechos.merge(ciudades_validas[['id_ciudad_sk'] + geografia],
                          on=geografia, how='left', validate='many_to_one')
    claves = ['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']
    rechazos = hechos.loc[hechos[claves].isna().any(axis=1)].copy()
    aceptados = hechos.dropna(subset=claves).copy()
    for campo in claves + ['numero_factura', 'numero_linea']:
        aceptados[campo] = aceptados[campo].astype(int)
    columnas = claves + [
        'numero_factura', 'numero_linea', 'cantidad', 'precio_unitario',
        'monto_neto', 'monto_impuesto', 'costo_estimado', 'margen_bruto'
    ]
    aceptados = aceptados[columnas]
    if aceptados.duplicated(['numero_factura', 'numero_linea']).any():
        raise ValueError('El grano factura + InvoiceLineID no es unico')
    return aceptados, rechazos, {'cliente': fallback_cliente, 'producto': fallback_producto}

,linea_id,factura_id,cliente_nk,producto_nk,cantidad,total,impuesto,cliente_sk,producto_sk,neto
0,1,100,1,10,2,24.20,4.20,101,201,20.00
1,2,101,1,20,-1,-6.05,-1.05,101,202,-5.00


In [ ]:
PERMITIR_VERSION_ACTUAL = True

sql_fuente = """
WITH cabeceras AS (
    SELECT s.*,
        ROW_NUMBER() OVER (
            PARTITION BY s.id_factura ORDER BY s.ultima_edicion DESC, s.id_raw DESC
        ) AS version_cabecera
    FROM staging.invoices s
    INNER JOIN raw.invoices r ON r.id_raw = s.id_raw AND r.lote_id = s.lote_id
    WHERE NOT EXISTS (
        SELECT 1 FROM quarantine.invoices q
        WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
    ) AND NOT EXISTS (
        SELECT 1 FROM raw.validacion_calidad_invoices v
        WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id AND v.es_critico = 1
    )
)
SELECT l.id_linea AS id_raw_linea,
       l.id_factura AS numero_factura,
       l.id_producto AS id_producto_nk,
       l.cantidad,
       l.neto AS monto_neto,
       l.impuesto AS monto_impuesto,
       l.total AS monto_total,
       c.id_cliente AS id_cliente_nk,
       c.id_vendedor AS id_vendedor_nk,
       c.fecha_factura,
       r.payload AS payload_linea
FROM staging.invoice_lines l
INNER JOIN raw.invoice_lines r ON r.id_raw = l.id_linea
LEFT JOIN cabeceras c ON c.id_factura = l.id_factura AND c.version_cabecera = 1
WHERE NOT EXISTS (
    SELECT 1 FROM quarantine.invoice_lines q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
) AND NOT EXISTS (
    SELECT 1 FROM raw.validacion_calidad v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id AND v.es_critico = 1
)
"""

with engine_destino.connect() as conn:
    df_fuente = pd.read_sql_query(text(sql_fuente), conn, coerce_float=False)
    dim_tiempo = pd.read_sql_query('SELECT id_tiempo FROM dw.dim_tiempo', conn)
    dim_cliente = pd.read_sql_query('''
        SELECT id_cliente_sk, id_cliente_nk, fecha_inicio_vigencia,
               fecha_fin_vigencia, es_actual, ciudad, estado_provincia, pais
        FROM dw.dim_cliente
    ''', conn)
    dim_producto = pd.read_sql_query('''
        SELECT id_producto_sk, id_producto_nk, fecha_inicio_vigencia,
               fecha_fin_vigencia, es_actual
        FROM dw.dim_producto
    ''', conn)
    dim_vendedor = pd.read_sql_query(
        'SELECT id_vendedor_sk, id_vendedor_nk FROM dw.dim_vendedor', conn
    )
    dim_ciudad = pd.read_sql_query('''
        SELECT id_ciudad_sk, ciudad, estado_provincia, pais
        FROM (
            SELECT id_ciudad_sk, ciudad, estado_provincia, pais,
                   COUNT(*) OVER (PARTITION BY ciudad, estado_provincia, pais) AS coincidencias
            FROM dw.dim_ciudad
        ) geografia
        WHERE coincidencias = 1
    ''', conn)

if df_fuente.empty:
    raise RuntimeError('No hay lineas elegibles: ejecutar extraccion, profiling, cuarentena y ambos stagings')

ventas, rechazos_cabecera = preparar_ventas(df_fuente)
df_fact_ok, rechazos_sk, usos_version_actual = construir_hechos(
    ventas, dim_tiempo, dim_cliente, dim_producto, dim_vendedor, dim_ciudad,
    permitir_actual=PERMITIR_VERSION_ACTUAL
)
print(f'Lineas de staging leidas: {len(df_fuente)}')
print(f'Lineas sin cabecera aceptada: {len(rechazos_cabecera)}')
print(f'Lineas sin alguna SK: {len(rechazos_sk)}')
print(f'Fallback SCD2 a version actual: {usos_version_actual}')
print(f'Hechos listos: {len(df_fact_ok)}')
print(f'Hechos sin margen disponible: {df_fact_ok["margen_bruto"].isna().sum()}')
if not rechazos_cabecera.empty:
    display(rechazos_cabecera[['numero_factura', 'numero_linea']].head(20))
if not rechazos_sk.empty:
    display(rechazos_sk[['numero_factura', 'numero_linea', 'id_cliente_sk',
                         'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk', 'id_tiempo']].head(20))
display(df_fact_ok.head(20))

{'neto': Decimal('15.00'), 'impuesto': Decimal('3.15'), 'total': Decimal('18.15')}
Filas de hechos: 2


## 2. Carga incremental y controles

- `cantidad` y `precio_unitario` se conservan; el precio unitario no es una medida aditiva.
- `monto_neto = total - impuesto`, usando los importes normalizados de staging. No se vuelve a considerar `ExtendedPrice` como neto porque incluye impuesto.
- `monto_impuesto` proviene de staging. Se verifica `neto + impuesto = total` con `Decimal`.
- `margen_bruto = LineProfit` y `costo_estimado = monto_neto - margen_bruto`. Si el payload no contiene `LineProfit`, ambos se cargan como NULL; no se inventan ni se reemplazan por cero.
- Las notas de credito aceptadas por el pipeline conservan cantidades e importes negativos.
- Se elige una cabecera aceptada por factura y una version aceptada por `InvoiceLineID`, evitando multiplicar hechos por lotes repetidos.
- La carga inserta solo claves nuevas `(numero_factura, numero_linea)` y verifica las cinco relaciones dimensionales.

**Tabla ya cargada:** el notebook original de referencia generaba `numero_linea` como ordinal dentro de la factura. Este pipeline usa el ID original para mantener una clave estable frente a cuarentena. Si encuentra numeracion incompatible o valores distintos en hechos existentes, se detiene sin sobrescribirlos: requieren una migracion o ajuste auditado separado.

Los rechazos por falta de cabecera o SK se muestran en la salida de la celda anterior y no se insertan. La carga no elimina hechos historicos existentes.

In [ ]:
from sqlalchemy import Numeric, text


def cargar_hechos(conn, candidatos):
    claves = ['numero_factura', 'numero_linea']
    existentes = pd.read_sql_query('SELECT * FROM dw.fact_ventas', conn, coerce_float=False)
    if existentes.duplicated(claves).any():
        raise ValueError('dw.fact_ventas ya contiene claves de negocio duplicadas')
    comparacion = candidatos.merge(
        existentes[list(candidatos.columns)], on=claves, how='left',
        suffixes=('', '_dw'), indicator='_carga', validate='one_to_one'
    )
    coincidentes = comparacion.loc[comparacion['_carga'] == 'both']
    for campo in candidatos.columns.difference(claves):
        distintos = (
            coincidentes[campo].notna()
            & ~coincidentes[campo].eq(coincidentes[f'{campo}_dw'])
        )
        if distintos.any():
            raise ValueError(f'Hechos existentes con cambios en {campo}; se requiere ajuste auditado, no se sobrescribe')
    nuevas = comparacion.loc[comparacion['_carga'] == 'left_only', candidatos.columns].copy()
    facturas_previas = set(existentes['numero_factura'])
    facturas_coincidentes = set(coincidentes['numero_factura'])
    ambiguas = set(nuevas['numero_factura']) & (facturas_previas - facturas_coincidentes)
    if ambiguas:
        raise ValueError('Hay facturas existentes sin ninguna linea coincidente. Revisar si usan numero_linea ordinal en vez de InvoiceLineID antes de continuar')
    tipos = {campo: Numeric(18, 4) for campo in ['cantidad', 'precio_unitario']}
    tipos.update({campo: Numeric(18, 2) for campo in [
        'monto_neto', 'monto_impuesto', 'costo_estimado', 'margen_bruto'
    ]})
    if not nuevas.empty:
        nuevas.to_sql('fact_ventas', conn, schema='dw', if_exists='append',
                      index=False, chunksize=500, dtype=tipos)
    return nuevas


sql_ddl_fact = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'dw')
    EXEC('CREATE SCHEMA dw');

IF OBJECT_ID('dw.fact_ventas', 'U') IS NULL
BEGIN
    CREATE TABLE dw.fact_ventas (
        id_fact_venta_sk BIGINT IDENTITY(1,1) PRIMARY KEY,
        id_tiempo INT NOT NULL,
        id_cliente_sk INT NOT NULL,
        id_producto_sk INT NOT NULL,
        id_vendedor_sk INT NOT NULL,
        id_ciudad_sk INT NOT NULL,
        numero_factura INT NOT NULL,
        numero_linea INT NOT NULL,
        cantidad DECIMAL(18,4) NOT NULL,
        precio_unitario DECIMAL(18,4) NOT NULL,
        monto_neto DECIMAL(18,2) NOT NULL,
        monto_impuesto DECIMAL(18,2) NULL,
        costo_estimado DECIMAL(18,2) NULL,
        margen_bruto DECIMAL(18,2) NULL,
        fecha_carga DATETIME2(7) NOT NULL DEFAULT SYSDATETIME(),
        CONSTRAINT uq_fact_ventas_doc_linea UNIQUE(numero_factura, numero_linea),
        CONSTRAINT fk_fact_ventas_tiempo FOREIGN KEY(id_tiempo) REFERENCES dw.dim_tiempo(id_tiempo),
        CONSTRAINT fk_fact_ventas_cliente FOREIGN KEY(id_cliente_sk) REFERENCES dw.dim_cliente(id_cliente_sk),
        CONSTRAINT fk_fact_ventas_producto FOREIGN KEY(id_producto_sk) REFERENCES dw.dim_producto(id_producto_sk),
        CONSTRAINT fk_fact_ventas_vendedor FOREIGN KEY(id_vendedor_sk) REFERENCES dw.dim_vendedor(id_vendedor_sk),
        CONSTRAINT fk_fact_ventas_ciudad FOREIGN KEY(id_ciudad_sk) REFERENCES dw.dim_ciudad(id_ciudad_sk)
    );
    CREATE INDEX ix_fact_ventas_tiempo ON dw.fact_ventas(id_tiempo);
    CREATE INDEX ix_fact_ventas_cliente ON dw.fact_ventas(id_cliente_sk);
    CREATE INDEX ix_fact_ventas_producto ON dw.fact_ventas(id_producto_sk);
    CREATE INDEX ix_fact_ventas_vendedor ON dw.fact_ventas(id_vendedor_sk);
    CREATE INDEX ix_fact_ventas_ciudad ON dw.fact_ventas(id_ciudad_sk);
END;
"""

sql_control_fk = """
SELECT COUNT(*) AS hechos_huerfanos
FROM dw.fact_ventas f
LEFT JOIN dw.dim_tiempo t ON t.id_tiempo = f.id_tiempo
LEFT JOIN dw.dim_cliente c ON c.id_cliente_sk = f.id_cliente_sk
LEFT JOIN dw.dim_producto p ON p.id_producto_sk = f.id_producto_sk
LEFT JOIN dw.dim_vendedor v ON v.id_vendedor_sk = f.id_vendedor_sk
LEFT JOIN dw.dim_ciudad ci ON ci.id_ciudad_sk = f.id_ciudad_sk
WHERE t.id_tiempo IS NULL OR c.id_cliente_sk IS NULL OR p.id_producto_sk IS NULL
   OR v.id_vendedor_sk IS NULL OR ci.id_ciudad_sk IS NULL
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_ddl_fact))
    df_nuevas = cargar_hechos(conn, df_fact_ok)
    huerfanos = conn.execute(text(sql_control_fk)).scalar_one()
    if huerfanos:
        raise ValueError(f'Hay {huerfanos} hechos huerfanos; carga revertida')

print(f'[OK] {len(df_nuevas)} hechos nuevos insertados en dw.fact_ventas')
print(f'[OK] {len(df_fact_ok) - len(df_nuevas)} hechos ya existentes, sin duplicarlos')
print('[OK] Cero FK huerfanas')
with engine_destino.connect() as conn:
    resumen = pd.read_sql_query('''
        SELECT COUNT(*) AS total_hechos, SUM(cantidad) AS cantidad_total,
               SUM(monto_neto) AS ventas_netas, SUM(monto_impuesto) AS impuestos,
               SUM(margen_bruto) AS margen_disponible
        FROM dw.fact_ventas
    ''', conn, coerce_float=False)
    muestra = pd.read_sql_query('''
        SELECT TOP 20 * FROM dw.fact_ventas ORDER BY id_fact_venta_sk DESC
    ''', conn, coerce_float=False)
display(resumen)
display(muestra)

In [ ]:
auditoria_etapa.finalizar(globals())